In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from plotting import *
setup_plotting_style()
print(f"saving figures:{SAVE_FIGURES}")

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

NB_ID = "21"

In [ ]:
print("Loading Metadata and Dataset...")

raw_path = get_unet_path(STAGE_SPLIT, split=TRAIN, signal=CommSignal2)
aug_path = get_unet_path(STAGE_AUGMENTED, split=TRAIN, signal=CommSignal2)
metadata_path = get_unet_path(STAGE_AUGMENTED, split=TRAIN, signal=CommSignal2, extension="csv")

# LOAD the actual data into memory (Crucial fix for your TypeError)
raw_CommSignal2_data = np.load(raw_path)
aug_dataset_data = np.load(aug_path)
df_meta = pd.read_csv(metadata_path)

print(f"Loaded Raw Data: {raw_CommSignal2_data.shape}")
print(f"Loaded Augmented Data: {aug_dataset_data.shape}")

In [ ]:
def plot_check(parent_id, df_meta, full_dataset):
    """
    Plots Identity vs. Rotated vs. Flipped for the same source file.
    """
    # Select samples for this parent
    subset = df_meta[df_meta['parent_id'] == parent_id]
    
    # Try to find one of each type
    try:
        row_ident = subset[subset['aug_type'] == 'Identity'].iloc[0]
        row_rot   = subset[subset['aug_type'] == 'Rotated'].iloc[0]
        row_flip  = subset[subset['aug_type'] == 'Rotated+Flipped'].iloc[0]
    except IndexError:
        print(f"Skipping Parent {parent_id}: Does not have all 3 types (Identity, Rotated, Flipped).")
        return

    # Extract Signals
    sig_ident = full_dataset[row_ident['sample_index']]
    sig_rot   = full_dataset[row_rot['sample_index']]
    sig_flip  = full_dataset[row_flip['sample_index']]
    
    # Visualization Setup (3 Rows x 2 Columns)
    fig, axes = plt.subplots(3, 2, figsize=(14, 15))
    fig.suptitle(f"Augmentation Check: Parent File #{parent_id}", y=0.95)

    # --- Row 1: Identity ---
    # Time Domain
    axes[0,0].plot(sig_ident[0, :500], label='I (Real)', alpha=0.8)
    axes[0,0].plot(sig_ident[1, :500], label='Q (Imag)', alpha=0.8)
    axes[0,0].set_title(f"Identity (Sample #{row_ident['sample_index']}) - Time Domain")
    axes[0,0].legend(loc='upper right')
    axes[0,0].grid(True, alpha=0.3)
    
    # Constellation
    axes[0,1].scatter(sig_ident[0, :5000], sig_ident[1, :5000], s=1, alpha=0.5, c='blue')
    axes[0,1].set_title("Identity - Constellation")
    axes[0,1].axis('equal')
    axes[0,1].grid(True)

    # --- Row 2: Rotated ---
    theta_rot = np.degrees(row_rot['theta'])
    
    # Time Domain
    axes[1,0].plot(sig_rot[0, :500], label='I (Real)', color='orange', alpha=0.8)
    axes[1,0].plot(sig_rot[1, :500], label='Q (Imag)', color='green', alpha=0.8)
    axes[1,0].set_title(f"Rotated Only (Sample #{row_rot['sample_index']}) - Time Domain")
    axes[1,0].text(0.02, 0.9, f"Rotation: {theta_rot:.1f}°", transform=axes[1,0].transAxes, fontsize=10, bbox=dict(facecolor='white', alpha=0.8))
    axes[1,0].legend(loc='upper right')
    axes[1,0].grid(True, alpha=0.3)
    
    # Constellation
    axes[1,1].scatter(sig_rot[0, :5000], sig_rot[1, :5000], s=1, alpha=0.5, c='orange')
    axes[1,1].set_title(f"Rotated ({theta_rot:.1f}°) - Constellation")
    axes[1,1].axis('equal')
    axes[1,1].grid(True)

    # --- Row 3: Rotated + Flipped ---
    theta_flip = np.degrees(row_flip['theta'])
    
    # Time Domain
    axes[2,0].plot(sig_flip[0, :500], label='I (Real)', color='purple', alpha=0.8)
    axes[2,0].plot(sig_flip[1, :500], label='Q (Imag)', color='brown', alpha=0.8)
    axes[2,0].set_title(f"Rotated + Flipped (Sample #{row_flip['sample_index']}) - Time Domain")
    axes[2,0].text(0.02, 0.9, f"Rotation: {theta_flip:.1f}° + Conjugate Flip", transform=axes[2,0].transAxes, fontsize=10, bbox=dict(facecolor='white', alpha=0.8))
    axes[2,0].legend(loc='upper right')
    axes[2,0].grid(True, alpha=0.3)
    
    # Constellation
    axes[2,1].scatter(sig_flip[0, :5000], sig_flip[1, :5000], s=1, alpha=0.5, c='red')
    axes[2,1].set_title(f"Rotated ({theta_flip:.1f}°) + Flipped - Constellation")
    axes[2,1].axis('equal')
    axes[2,1].grid(True)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    save_plot(f"aug_check_parent_{parent_id:03d}", machine_id="B", nb_id=NB_ID, fig_id="01")
    plt.show()

In [ ]:
# Find parents that have at least one of each type
parents_with_all = []
for pid in df_meta['parent_id'].unique():
    types = df_meta[df_meta['parent_id'] == pid]['aug_type'].unique()
    if 'Identity' in types and 'Rotated' in types and 'Rotated+Flipped' in types:
        parents_with_all.append(pid)

if not parents_with_all:
    print("Could not find a single parent with all 3 types! (This is statistically unlikely but possible).")
else:
    print(f"Found {len(parents_with_all)} parents with full augmentation spread.")
    random_parent = np.random.choice(parents_with_all)
    plot_check(random_parent, df_meta, aug_dataset_data)

In [ ]:
NUM_SAMPLES_TO_TEST = 100  # Number of random samples to forensically verify

def calculate_complex_correlation(sig1, sig2):
    """
    Computes the complex correlation coefficient between two signals.
    Returns: (Magnitude, Angle_Difference)
    """
    # Convert to complex
    z1 = sig1[0] + 1j * sig1[1]
    z2 = sig2[0] + 1j * sig2[1]
    
    # Dot product
    correlation = np.vdot(z1, z2) / (np.linalg.norm(z1) * np.linalg.norm(z2))
    
    return np.abs(correlation), np.angle(correlation)

print("-" * 60)
print(f"Selecting {NUM_SAMPLES_TO_TEST} random 'Rotated' samples to verify...")

# Filter only for samples that were Rotated (ignoring Identity for now)
rotated_subset = df_meta[df_meta['aug_type'] == 'Rotated'].sample(n=NUM_SAMPLES_TO_TEST, random_state=42)

results = []

for _, row in tqdm(rotated_subset.iterrows(), total=NUM_SAMPLES_TO_TEST):
    # Get the Augmented Data (Result)
    aug_sample = aug_dataset_data[row['sample_index']]
    
    # Re-create the Original Data (Ground Truth)
    parent_id = row['parent_id']
    start = row['start_idx']
    length = aug_sample.shape[1] # 40960
    
    original_slice = raw_CommSignal2_data[parent_id, :, start : start + length]
    
    # Calculate Physics Metrics
    
    # A. Amplitude Preservation Check
    # RMS (Root Mean Square) energy of both signals
    energy_orig = np.sqrt(np.mean(original_slice**2))
    energy_aug  = np.sqrt(np.mean(aug_sample**2))
    
    # Amplitude Error (Should be 0.0)
    ampl_error_percent = abs(energy_aug - energy_orig) / energy_orig * 100.0
    
    # B. Phase Accuracy Check
    # Measure the actual rotation between Original and Augmented
    measured_corr, measured_angle = calculate_complex_correlation(original_slice, aug_sample)
    
    # The measured angle should match the 'theta' in metadata
    expected_theta = row['theta']
    
    # Wrap error to [-pi, pi] to handle 360-degree crossovers
    phase_error_rads = np.arctan2(np.sin(measured_angle - expected_theta), 
                                  np.cos(measured_angle - expected_theta))
    
    results.append({
        'sample_index': row['sample_index'],
        'ampl_error_pct': ampl_error_percent,
        'phase_error_deg': np.degrees(np.abs(phase_error_rads)),
        'correlation_magnitude': measured_corr
    })


# --- Report Generation ---
df_res = pd.DataFrame(results)

print("\n" + "="*40)
print("VERIFICATION RESULTS")
print("="*40)

# Signal Integrity (Amplitude)
print(f"Amplitude Preservation Error (Mean): {df_res['ampl_error_pct'].mean():.6f} %")
print(f"Amplitude Preservation Error (Max):  {df_res['ampl_error_pct'].max():.6f} %")
print("   -> PASS if < 1e-4 %")

# Augmentation Accuracy (Phase)
print(f"\nRotation Angle Error (Mean):       {df_res['phase_error_deg'].mean():.6f} degrees")
print(f"Rotation Angle Error (Max):        {df_res['phase_error_deg'].max():.6f} degrees")
print("   -> PASS if < 1e-2 degrees")

# Correlation (Shape Similarity)
print(f"\nStructural Correlation (Mean):     {df_res['correlation_magnitude'].mean():.6f}")
print("   -> PASS if > 0.999999 (1.0 means perfect shape match)")

if df_res['ampl_error_pct'].mean() < 1e-3 and df_res['correlation_magnitude'].mean() > 0.9999:
    print("\nVERIFICATION SUCCESSFUL: Augmentation is mathematically precise.")
else:
    print("\nVERIFICATION FAILED: Significant distortions detected.")